# M6CifarMlpNotebook

这个 Notebook 由可视化模型图自动生成，按功能拆成多个小节，便于逐段阅读、运行和改写。

- 层数量：5
- 连接数量：4
- 输入维度：[[3072]]
- 输出维度：[[10]]


## 1. 依赖导入

##### 功能
导入 PyTorch 和神经网络模块。后面的模型类、辅助层和运行示例都依赖这些包。


In [ ]:
import argparse
import os
import torch
import torch.nn as nn
import torch.utils.data
import torchvision


## 2. 数据集与训练配置

##### 功能
这里保存从模型 JSON 导出的训练配置，包括数据集、批大小、训练轮数、学习率、优化器和损失函数。后续 DataLoader 和训练循环会直接读取这些配置。


In [ ]:
TRAIN_CONFIG = {'dataset_name': 'CIFAR10', 'epochs': 1, 'batch_size': 1, 'rate': 0.001, 'device': 'cpu', 'loss_fn': 'cross_entropy', 'optimizer': 'sgd', 'data_dir': '', 'artifacts_dir': ''}

DATASET_SPECS = {
    'MNIST': {'class': torchvision.datasets.MNIST, 'shape': [1, 28, 28], 'classes': 10},
    'FashionMNIST': {'class': torchvision.datasets.FashionMNIST, 'shape': [1, 28, 28], 'classes': 10},
    'KMNIST': {'class': torchvision.datasets.KMNIST, 'shape': [1, 28, 28], 'classes': 10},
    'CIFAR10': {'class': torchvision.datasets.CIFAR10, 'shape': [3, 32, 32], 'classes': 10},
    'CIFAR100': {'class': torchvision.datasets.CIFAR100, 'shape': [3, 32, 32], 'classes': 100},
}


In [ ]:
MODEL_INPUTS = [{'id': 'input', 'shape': [3072]}]
MODEL_INPUT_SHAPES = [item['shape'] for item in MODEL_INPUTS]


## 3. 数据集加载

##### 功能
根据 `TRAIN_CONFIG['dataset_name']` 加载 torchvision 内置数据集，并按数据集类型选择输入转换。这一步把导出的模型和真实数据连接起来。


In [ ]:
def resolve_dataset_name(dataset_name):
    aliases = {
        "mnist": "MNIST",
        "fashionmnist": "FashionMNIST",
        "fashion_mnist": "FashionMNIST",
        "fashion-mnist": "FashionMNIST",
        "kmnist": "KMNIST",
        "cifar10": "CIFAR10",
        "cifar-10": "CIFAR10",
        "cifar_10": "CIFAR10",
        "cifar100": "CIFAR100",
        "cifar-100": "CIFAR100",
        "cifar_100": "CIFAR100",
    }
    normalized = str(dataset_name or "MNIST").strip()
    if normalized in DATASET_SPECS:
        return normalized
    key = aliases.get(normalized.lower())
    if key:
        return key
    supported = ", ".join(DATASET_SPECS)
    raise ValueError(f"Unsupported dataset: {dataset_name}. Supported datasets: {supported}")


def get_primary_model_input_shape(model_inputs=None):
    model_inputs = model_inputs or MODEL_INPUTS
    if not model_inputs:
        dataset_name = resolve_dataset_name(TRAIN_CONFIG.get("dataset_name", "MNIST"))
        return DATASET_SPECS[dataset_name]["shape"]
    return list(model_inputs[0].get("shape", []))


def _product(values):
    result = 1
    for value in values:
        result *= int(value)
    return result


def build_dataset_transform(dataset_name, model_inputs=None):
    dataset_shape = DATASET_SPECS[dataset_name]["shape"]
    target_shape = get_primary_model_input_shape(model_inputs)
    transforms = []

    if len(target_shape) == 3:
        target_channels, target_height, target_width = target_shape
        if target_channels not in (1, 3):
            raise ValueError(f"Image dataset export only supports 1 or 3 input channels, got {target_channels}")
        transforms.append(torchvision.transforms.Resize((target_height, target_width)))
        if target_channels != dataset_shape[0]:
            transforms.append(torchvision.transforms.Grayscale(num_output_channels=target_channels))
        transforms.append(torchvision.transforms.ToTensor())
        return torchvision.transforms.Compose(transforms)

    if len(target_shape) == 2:
        target_height, target_width = target_shape
        transforms.extend([
            torchvision.transforms.Grayscale(num_output_channels=1),
            torchvision.transforms.Resize((target_height, target_width)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Lambda(lambda tensor: tensor.squeeze(0)),
        ])
        return torchvision.transforms.Compose(transforms)

    if len(target_shape) == 1:
        target_features = int(target_shape[0])
        dataset_features = _product(dataset_shape)
        if target_features == dataset_features:
            transforms.extend([
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Lambda(lambda tensor: torch.flatten(tensor)),
            ])
        else:
            transforms.extend([
                torchvision.transforms.Grayscale(num_output_channels=1),
                torchvision.transforms.Resize((1, target_features)),
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Lambda(lambda tensor: torch.flatten(tensor)),
            ])
        return torchvision.transforms.Compose(transforms)

    raise ValueError(f"Unsupported model input shape for image dataset: {target_shape}")


def prepare_dataloaders(config):
    dataset_name = resolve_dataset_name(config.get("dataset_name", "MNIST"))
    dataset_class = DATASET_SPECS[dataset_name]["class"]
    batch_size = int(config.get("batch_size", 64))
    data_dir = os.path.expanduser(config.get("data_dir") or "./datasets")
    dataset_root = os.path.join(data_dir, dataset_name)
    transform = build_dataset_transform(dataset_name, MODEL_INPUTS)

    train_data = dataset_class(
        root=dataset_root,
        train=True,
        transform=transform,
        download=True,
    )
    test_data = dataset_class(
        root=dataset_root,
        train=False,
        transform=transform,
        download=True,
    )

    train_loader = torch.utils.data.DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
    )
    test_loader = torch.utils.data.DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
    )
    return train_loader, test_loader


def build_sample_input_from_dataset(config, model_inputs=None):
    dataset_name = resolve_dataset_name(config.get("dataset_name", "MNIST"))
    dataset_shape = DATASET_SPECS[dataset_name]["shape"]
    model_inputs = model_inputs or []
    input_shapes = [item.get("shape", []) for item in model_inputs]
    shape = input_shapes[0] if input_shapes else dataset_shape
    batch_size = int(config.get("batch_size", 64))
    if len(model_inputs) > 1:
        return {
            item["id"]: torch.randn(batch_size, *item.get("shape", []))
            for item in model_inputs
        }
    return torch.randn(batch_size, *shape)


## 4. 高级辅助层

##### 功能
这里放置序列、注意力、VAE、图卷积等高级模块的封装。当前模型未用到的辅助类也可以保留，方便继续扩展画布。


In [ ]:
class SelfAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

    def forward(self, x):
        output, _ = self.attention(x, x, x)
        return output


class LSTMLayer(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, bidirectional=False, return_sequences=False):
        super().__init__()
        self.return_sequences = return_sequences
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            batch_first=True,
        )

    def forward(self, x):
        output, _ = self.lstm(x)
        if self.return_sequences:
            return output
        return output[:, -1, :]


class Seq2SeqLayer(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, target_length, num_layers=1):
        super().__init__()
        self.target_length = target_length
        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.decoder_cell = nn.LSTM(
            input_size=output_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.initial_decoder_input = nn.Parameter(torch.zeros(1, 1, output_size))
        self.output_projection = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        batch_size = x.size(0)
        _, hidden = self.encoder(x)
        decoder_input = self.initial_decoder_input.expand(batch_size, self.target_length, -1)
        decoder_output, _ = self.decoder_cell(decoder_input, hidden)
        return self.output_projection(decoder_output)


class VAELayer(nn.Module):
    def __init__(self, input_features, latent_dim, output_features):
        super().__init__()
        self.encoder_mu = nn.Linear(input_features, latent_dim)
        self.encoder_logvar = nn.Linear(input_features, latent_dim)
        self.decoder = nn.Linear(latent_dim, output_features)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        mu = self.encoder_mu(x)
        logvar = self.encoder_logvar(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return self.decoder(z)


class GraphConvLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.projection = nn.Linear(in_features, out_features)

    def forward(self, x):
        adjacency = None
        if isinstance(x, dict):
            adjacency = x.get("adj")
            x = x.get("x")

        support = self.projection(x)
        if adjacency is None:
            return support
        return torch.matmul(adjacency, support)


## 5. 模型主体：`M6CifarMlpNotebook`

##### 功能
这一段是从画布连接关系生成的 `nn.Module`。`__init__` 定义每个可计算层，`forward` 按拓扑顺序执行数据流。


In [ ]:
class M6CifarMlpNotebook(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_fc1 = nn.Linear(in_features=3072, out_features=128)
        self.layer_relu = nn.ReLU()
        self.layer_classifier = nn.Linear(in_features=128, out_features=10)

    def forward(self, x):
        outputs = {}
        out_input = x
        outputs['input'] = out_input
        out_fc1 = self.layer_fc1(outputs['input'])
        outputs['fc1'] = out_fc1
        out_relu = self.layer_relu(outputs['fc1'])
        outputs['relu'] = out_relu
        out_classifier = self.layer_classifier(outputs['relu'])
        outputs['classifier'] = out_classifier
        out_out = outputs['classifier']
        outputs['out'] = out_out
        return outputs['out']


## 6. 结构与维度总览

##### 功能
用表格化数据查看每一层的类型、输入维度、输出维度和关键参数，帮助确认画布结构是否符合预期。


In [ ]:
layer_summaries = [{'id': 'input', 'type': 'Input', 'input_shape': None, 'output_shape': [3072], 'params': {'shape': [3072]}}, {'id': 'fc1', 'type': 'Linear', 'input_shape': [3072], 'output_shape': [128], 'params': {'out_features': 128}}, {'id': 'relu', 'type': 'ReLU', 'input_shape': [128], 'output_shape': [128], 'params': {}}, {'id': 'classifier', 'type': 'Linear', 'input_shape': [128], 'output_shape': [10], 'params': {'out_features': 10}}, {'id': 'out', 'type': 'Output', 'input_shape': [10], 'output_shape': [10], 'params': {}}]
for index, item in enumerate(layer_summaries, start=1):
    print(f"{index}. {item['id']} ({item['type']})")
    print(f"   input : {item['input_shape']}")
    print(f"   output: {item['output_shape']}")
    print(f"   params: {item['params']}")


### 模块 1: `input` (Input)

##### 功能
声明模型接收的数据形状。运行时会把传入的张量作为后续层的数据源。

##### 维度
- 输入维度：`None`
- 输出维度：`[3072]`


In [ ]:
module_info = {'id': 'input', 'type': 'Input', 'params': {'shape': [3072]}, 'input_shape': None, 'output_shape': [3072], 'status': 'ok', 'note': '声明模型接收的数据形状。运行时会把传入的张量作为后续层的数据源。'}
print(f"模块: {module_info['id']} ({module_info['type']})")
print(f"功能: {module_info['note']}")
print(f"输入维度: {module_info['input_shape']}")
print(f"输出维度: {module_info['output_shape']}")
print(f"关键参数: {module_info['params']}")


### 模块 2: `fc1` (Linear)

##### 功能
全连接层，对输入特征做线性变换。本层输出特征数为 128。

##### 维度
- 输入维度：`[3072]`
- 输出维度：`[128]`


In [ ]:
module_info = {'id': 'fc1', 'type': 'Linear', 'params': {'out_features': 128}, 'input_shape': [3072], 'output_shape': [128], 'status': 'ok', 'note': '全连接层，对输入特征做线性变换。本层输出特征数为 128。'}
print(f"模块: {module_info['id']} ({module_info['type']})")
print(f"功能: {module_info['note']}")
print(f"输入维度: {module_info['input_shape']}")
print(f"输出维度: {module_info['output_shape']}")
print(f"关键参数: {module_info['params']}")


### 模块 3: `relu` (ReLU)

##### 功能
ReLU 激活层，把负值截断为 0，引入非线性表达能力。

##### 维度
- 输入维度：`[128]`
- 输出维度：`[128]`


In [ ]:
module_info = {'id': 'relu', 'type': 'ReLU', 'params': {}, 'input_shape': [128], 'output_shape': [128], 'status': 'ok', 'note': 'ReLU 激活层，把负值截断为 0，引入非线性表达能力。'}
print(f"模块: {module_info['id']} ({module_info['type']})")
print(f"功能: {module_info['note']}")
print(f"输入维度: {module_info['input_shape']}")
print(f"输出维度: {module_info['output_shape']}")
print(f"关键参数: {module_info['params']}")


### 模块 4: `classifier` (Linear)

##### 功能
全连接层，对输入特征做线性变换。本层输出特征数为 10。

##### 维度
- 输入维度：`[128]`
- 输出维度：`[10]`


In [ ]:
module_info = {'id': 'classifier', 'type': 'Linear', 'params': {'out_features': 10}, 'input_shape': [128], 'output_shape': [10], 'status': 'ok', 'note': '全连接层，对输入特征做线性变换。本层输出特征数为 10。'}
print(f"模块: {module_info['id']} ({module_info['type']})")
print(f"功能: {module_info['note']}")
print(f"输入维度: {module_info['input_shape']}")
print(f"输出维度: {module_info['output_shape']}")
print(f"关键参数: {module_info['params']}")


### 模块 5: `out` (Output)

##### 功能
标记模型的最终输出位置。该节点不改变张量，只把前一层结果作为模型返回值。

##### 维度
- 输入维度：`[10]`
- 输出维度：`[10]`


In [ ]:
module_info = {'id': 'out', 'type': 'Output', 'params': {}, 'input_shape': [10], 'output_shape': [10], 'status': 'ok', 'note': '标记模型的最终输出位置。该节点不改变张量，只把前一层结果作为模型返回值。'}
print(f"模块: {module_info['id']} ({module_info['type']})")
print(f"功能: {module_info['note']}")
print(f"输入维度: {module_info['input_shape']}")
print(f"输出维度: {module_info['output_shape']}")
print(f"关键参数: {module_info['params']}")


## 7. 训练与评估函数

##### 功能
定义一个完整的训练轮次、评估函数和 `run_training` 入口。运行训练时会使用上面配置的数据集、损失函数和优化器。


In [ ]:
def build_loss_fn(config):
    loss_name = str(config.get("loss_fn", "cross_entropy")).lower()
    if loss_name in ("cross_entropy", "crossentropyloss", "ce"):
        return nn.CrossEntropyLoss()
    if loss_name in ("mse", "mseloss"):
        return nn.MSELoss()
    raise ValueError(f"Unsupported loss function: {config.get('loss_fn')}")


def build_optimizer(model, config):
    optimizer_name = str(config.get("optimizer", "sgd")).lower()
    learning_rate = float(config.get("rate", 0.001))
    if optimizer_name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    if optimizer_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=learning_rate)
    raise ValueError(f"Unsupported optimizer: {config.get('optimizer')}")


def train_one_epoch(model, train_loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return {
        "loss": total_loss / total if total else 0.0,
        "accuracy": correct / total if total else 0.0,
    }


def evaluate(model, test_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += batch_size

    return {
        "loss": total_loss / total if total else 0.0,
        "accuracy": correct / total if total else 0.0,
    }


def run_training(model, config=None):
    config = dict(TRAIN_CONFIG if config is None else config)
    requested_device = config.get("device", "cpu")
    if requested_device == "cuda" and not torch.cuda.is_available():
        requested_device = "cpu"
    device = torch.device(requested_device)

    model = model.to(device)
    train_loader, test_loader = prepare_dataloaders(config)
    loss_fn = build_loss_fn(config)
    optimizer = build_optimizer(model, config)
    epochs = int(config.get("epochs", 1))

    history = []
    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        eval_metrics = evaluate(model, test_loader, loss_fn, device)
        item = {"epoch": epoch, "train": train_metrics, "eval": eval_metrics}
        history.append(item)
        print(
            f"epoch={epoch} "
            f"train_loss={train_metrics['loss']:.4f} "
            f"train_acc={train_metrics['accuracy']:.4f} "
            f"eval_loss={eval_metrics['loss']:.4f} "
            f"eval_acc={eval_metrics['accuracy']:.4f}"
        )
    return history


## 8. 前向传播试运行

##### 功能
根据当前数据集的输入形状构造一份样例输入，执行模型前向传播，并打印最终输出维度。它不会下载数据集或训练模型，只用于确认模型结构可以运行。


In [ ]:
model = M6CifarMlpNotebook()
sample_input = build_sample_input_from_dataset(TRAIN_CONFIG, MODEL_INPUTS)
output = model(sample_input)
print(model)
print('dataset:', TRAIN_CONFIG.get('dataset_name', 'MNIST'))
if isinstance(output, dict):
    print({key: tuple(value.shape) for key, value in output.items()})
else:
    print(tuple(output.shape))


## 9. 使用真实数据集训练

##### 功能
取消下面代码的注释后，会根据 `TRAIN_CONFIG` 下载/读取数据集并训练模型。首次运行某个数据集时可能需要等待下载完成。


In [ ]:
# history = run_training(model, TRAIN_CONFIG)
# history[-1] if history else None
